## Setup

In [1]:
import sys 
import os
import IPython


# Env. vars to check if the notebook is running on colab, kaggle etc. 
IS_COLAB = "google.colab" in sys.modules 
IS_KAGGLE = "kaggle_secrets" in sys.modules 
IS_LOCAL = not (IS_COLAB or IS_KAGGLE)

if IS_COLAB:
    # Install the packages 
    %pip install -q -U tensorflow-addons
    %pip install -q -U transformers
    %pip install -q -U datasets
    print("You can safely ignore the package incompatibility errors.")
    # Mount the drive 
    from google.colab import drive 
    drive.mount("/drive")

# NOTE: This needs to be done for correct notebook behavior with the transformers 
# library 
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [2]:
import pandas as pd
import numpy as np
from tqdm import tqdm 

import random 
import shutil 
# Scikit-Learn ≥0.20 is required
import sklearn
assert sklearn.__version__ >= "0.20"

# Pytorch imports 
import torch
import torch.nn as nn 
from torch.utils.data import Dataset, DataLoader

from typing import List, Tuple

# To plot pretty figures
import seaborn as sns
%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rc('axes', labelsize=14)
mpl.rc('xtick', labelsize=12)
mpl.rc('ytick', labelsize=12)

sns.set_theme()

In [3]:
# Project Paths
NOTEBOOK_NAME = "1.0-POC"
PROJECT_ROOT_DIR = "."

# --- Result dirs. 
OUTPUT_DIR = os.path.join(PROJECT_ROOT_DIR,"outputs",NOTEBOOK_NAME)
os.makedirs(OUTPUT_DIR,exist_ok=True)


LOGGING_DIR = os.path.join(OUTPUT_DIR,"logs")
os.makedirs(LOGGING_DIR,exist_ok=True)

In [4]:
# Setup logging for this notebook 
from LLMDialogue.utils.logging import configure_logger, create_file_handler, create_stream_handler
from LLMDialogue.utils.general import get_system_device
import logging

configure_logger(
    handlers=[
        create_stream_handler(logging_level=logging.WARNING), 
        create_file_handler(
            outdir=LOGGING_DIR, 
            logging_level=logging.DEBUG, 
            reset_outdir=False # Keep logs across runs. 
        )
    ]
)


In [5]:

GLOBAL_DEVICE = get_system_device()
# Shared env. vars. 
GLOBAL_SEED = 42 
RESET_OUTDIR = True
# Configuring env.
if  GLOBAL_SEED is not None:
    # to make this notebook's output stable across runs
    np.random.seed(GLOBAL_SEED) 
    torch.manual_seed(GLOBAL_SEED)
# NOTE: Should be mps for mac. 
GLOBAL_DEVICE

device(type='mps')

In [6]:
#Reset all but the logs dir in the output folder
from LLMDialogue.utils.general import remove_dir, get_subdirs_in_dir
if RESET_OUTDIR:
    subdirs = get_subdirs_in_dir(OUTPUT_DIR)
    for subdir in subdirs:
        if not subdir in ['logs']:
            remove_dir(subdir)


## Datasets

### In Conversation Corpus (ICC) 

#### Variant - No Labels

This variant of the ICC does not include any conversation start or end or 
speaker labels. 

In [7]:
from LLMDialogue.datasets import load
from LLMDialogue.datasets.ICC import objectify as ICCObjectify, ICCDatasetSplits

In [8]:
dset = load(
    dset_name="ICC", 
    config_name="no_labels"
)
print(dset)

{
    "date_created": "23_05_2024__13_16_57",
    "dset_name": "ICC",
    "variant_name": "no_labels",
    "version": "0.0.1",
    "data_dir": null,
    "data_files": {
        "train": "/Users/muhammadumair/Documents/Repositories/mumair01-repos/GPT-Monologue-to-Dialogue/data/raw/ICC/28_train_14_test_set/train",
        "test": "/Users/muhammadumair/Documents/Repositories/mumair01-repos/GPT-Monologue-to-Dialogue/data/raw/ICC/28_train_14_test_set/test"
    },
    "force_regenerate": false,
    "config_id": "6177b8867f970aef5e648c412b2434014347ee7638037e970b581a5c4942b019",
    "cache_dir": "/Users/muhammadumair/Documents/Repositories/mumair01-repos/GPT-Monologue-to-Dialogue/cache/datasets/6177b8867f970aef5e648c412b2434014347ee7638037e970b581a5c4942b019",
    "archive_dir": "/Users/muhammadumair/Documents/Repositories/mumair01-repos/GPT-Monologue-to-Dialogue/cache/datasets/6177b8867f970aef5e648c412b2434014347ee7638037e970b581a5c4942b019/archive",
    "compress_archive": true,
    "seed":

In [9]:
dset_splits = ICCObjectify(dset)
dset_splits.split_names


['train', 'test']

In [10]:
dset_splits.data_df_map['train']

,conversationName,conversationID,turnNumber,utterance
0,2018-02-04-session-1_nogaps_step1_GPT,0,0,all right
1,2018-02-04-session-1_nogaps_step1_GPT,0,1,hel
2,2018-02-04-session-1_nogaps_step1_GPT,0,2,hela hela this is go so weird to get used to
3,2018-02-04-session-1_nogaps_step1_GPT,0,3,i know i feel kinda
4,2018-02-04-session-1_nogaps_step1_GPT,0,4,all right um when's your first softball game
...,...,...,...,...
16825,2018-10-20-session-3_nogaps_step1_GPT,27,453,i'm like no high school junior or senior looke...
16826,2018-10-20-session-3_nogaps_step1_GPT,27,454,you<ts> i know exactly
16827,2018-10-20-session-3_nogaps_step1_GPT,27,455,not at my school
16828,2018-10-20-session-3_nogaps_step1_GPT,27,456,like it the the freshmen look like literal chi...


In [11]:
# Save the data dfs 
dset_name = dset.info().to_dict()['configs']['dset_name']
variant_name = dset.info().to_dict()['configs']['variant_name']
dset_splits.to_csv(
    outdir=f"{OUTPUT_DIR}/{dset_name}/{variant_name}"
)


{'train': './outputs/1.0-POC/ICC/no_labels/train.csv',
 'test': './outputs/1.0-POC/ICC/no_labels/test.csv'}

#### Variant - Special Labels

In [12]:
dset = load(
    dset_name="ICC", 
    config_name="labels"
)
print(dset)

{
    "date_created": "23_05_2024__13_16_58",
    "dset_name": "ICC",
    "variant_name": "labels",
    "version": "0.0.1",
    "data_dir": null,
    "data_files": {
        "train": "/Users/muhammadumair/Documents/Repositories/mumair01-repos/GPT-Monologue-to-Dialogue/data/raw/ICC/28_train_14_test_set/train",
        "test": "/Users/muhammadumair/Documents/Repositories/mumair01-repos/GPT-Monologue-to-Dialogue/data/raw/ICC/28_train_14_test_set/test"
    },
    "force_regenerate": false,
    "config_id": "16ca6817ab14e9b07e5907a0cf4231185131ca618d61c75a1d47fe4cd7759a2c",
    "cache_dir": "/Users/muhammadumair/Documents/Repositories/mumair01-repos/GPT-Monologue-to-Dialogue/cache/datasets/16ca6817ab14e9b07e5907a0cf4231185131ca618d61c75a1d47fe4cd7759a2c",
    "archive_dir": "/Users/muhammadumair/Documents/Repositories/mumair01-repos/GPT-Monologue-to-Dialogue/cache/datasets/16ca6817ab14e9b07e5907a0cf4231185131ca618d61c75a1d47fe4cd7759a2c/archive",
    "compress_archive": true,
    "seed": 42

In [13]:
dset_splits = ICCObjectify(dset)
dset_splits.split_names

['train', 'test']

In [14]:
dset_splits.data_df_map['train']

,conversationName,conversationID,turnNumber,utterance
0,2018-02-04-session-1_nogaps_step1_GPT,0,0,<START>
1,2018-02-04-session-1_nogaps_step1_GPT,0,1,<SP1> all right <SP1>
2,2018-02-04-session-1_nogaps_step1_GPT,0,2,<SP2> hel <SP2>
3,2018-02-04-session-1_nogaps_step1_GPT,0,3,<SP1> hela hela this is go so weird to get use...
4,2018-02-04-session-1_nogaps_step1_GPT,0,4,<SP2> i know i feel kinda <SP2>
...,...,...,...,...
21154,2018-10-20-session-3_nogaps_step1_GPT,27,567,<SP1> not at my school <SP1>
21155,2018-10-20-session-3_nogaps_step1_GPT,27,568,<SP2> like it the the freshmen look like liter...
21156,2018-10-20-session-3_nogaps_step1_GPT,27,569,<SP1> yeah the freshmen look like children and...
21157,2018-10-20-session-3_nogaps_step1_GPT,27,570,<SP1> like men <SP1>


In [15]:
# Save the data dfs 
dset_name = dset.info().to_dict()['configs']['dset_name']
variant_name = dset.info().to_dict()['configs']['variant_name']
dset_splits.to_csv(
    outdir=f"{OUTPUT_DIR}/{dset_name}/{variant_name}"
)


{'train': './outputs/1.0-POC/ICC/labels/train.csv',
 'test': './outputs/1.0-POC/ICC/labels/test.csv'}

### Speaker Identity

#### Variant - No Labels

In [16]:
from LLMDialogue.datasets.speakerIdentity import SpeakerIdentityDataSplits, objectify as SIObjectify

In [17]:
dset = load(
    dset_name="speakerIdentity",
    config_name="labels"
)

In [18]:
dset_splits = SIObjectify(dset)
dset_splits.split_names

['test']

In [19]:
dset_splits.data_df_map['test']

,conversationName,conversationID,turnNumber,utterance
0,30b,0,0,<START>
1,30b,0,1,<SP1> i haven't seen the keys anywhere <SP1>
2,30b,0,2,<SP2> have you <SP2>
3,30b,0,3,<END>
4,101bf,1,0,<START>
...,...,...,...,...
2995,18bd,748,3,<END>
2996,4bf,749,0,<START>
2997,4bf,749,1,<SP1> it's great that you could cover this shi...
2998,4bf,749,2,<SP1> for real <SP1>


In [20]:
# Save the data dfs 
dset_name = dset.info().to_dict()['configs']['dset_name']
variant_name = dset.info().to_dict()['configs']['variant_name']
dset_splits.to_csv(
    outdir=f"{OUTPUT_DIR}/{dset_name}/{variant_name}"
)


{'test': './outputs/1.0-POC/speakerIdentity/labels/test.csv'}

#### Labels

In [21]:
dset = load(
    dset_name="speakerIdentity",
    config_name="labels"
)

In [22]:
dset_splits = SIObjectify(dset)
dset_splits.split_names

['test']

In [23]:
dset_splits.data_df_map['test']

,conversationName,conversationID,turnNumber,utterance
0,30b,0,0,<START>
1,30b,0,1,<SP1> i haven't seen the keys anywhere <SP1>
2,30b,0,2,<SP2> have you <SP2>
3,30b,0,3,<END>
4,101bf,1,0,<START>
...,...,...,...,...
2995,18bd,748,3,<END>
2996,4bf,749,0,<START>
2997,4bf,749,1,<SP1> it's great that you could cover this shi...
2998,4bf,749,2,<SP1> for real <SP1>


In [24]:
# Save the data dfs 
dset_name = dset.info().to_dict()['configs']['dset_name']
variant_name = dset.info().to_dict()['configs']['variant_name']
dset_splits.to_csv(
    outdir=f"{OUTPUT_DIR}/{dset_name}/{variant_name}"
)

{'test': './outputs/1.0-POC/speakerIdentity/labels/test.csv'}

### Data Modules

These are required to be able to train the model 

In [25]:
from LLMDialogue.dm.ICCdm import ICCHFLMDM
from LLMDialogue.dm.HFdm import HFLanguageModelingDM
from transformers import AutoTokenizer
from LLMDialogue.dm.ICCdm import ICCSpokenLanguageModelingDM
from LLMDialogue.tokenizers import SpokenDialogueTokenizer

#### ICC Language Modeling Data Module (Compatible with HuggingFace)

In [26]:
# Load the tokenizer here
tokenizer = AutoTokenizer.from_pretrained("gpt2")

In [27]:
# Load the dm and prepare it.
dm = ICCHFLMDM(
    dset_variant="labels", 
    tokenizer=tokenizer, 
    save_dir=OUTPUT_DIR
)
dm.prepare_data()
dm.setup('fit')

Map:   0%|          | 0/21159 [00:00<?, ? examples/s]

Map:   0%|          | 0/10559 [00:00<?, ? examples/s]

Map:   0%|          | 0/21159 [00:00<?, ? examples/s]

Map:   0%|          | 0/10559 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/2675 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1298 [00:00<?, ? examples/s]

#### ICC Spoken Language Modeling Data Module (Compatible with PL)

In [28]:
tokenizer = SpokenDialogueTokenizer()

Additional special tokens map:
	bos_token: <|endoftext|>
	eos_token: <ts>
	unk_token: <|endoftext|>
	pad_token: <|endoftext|>
	additional_special_tokens: ['<SP1>', '<SP2>']

NOTE: This tokenizer assumes that there are 2 speakers


In [29]:
dm = ICCSpokenLanguageModelingDM(
    dset_variant="labels", 
    tokenizer=tokenizer,
    save_dir=OUTPUT_DIR
)
dm.prepare_data()
dm.setup('fit')

Map:   0%|          | 0/21159 [00:00<?, ? examples/s]

Map:   0%|          | 0/743 [00:00<?, ? examples/s]

Map:   0%|          | 0/782 [00:00<?, ? examples/s]

Map:   0%|          | 0/802 [00:00<?, ? examples/s]

Map:   0%|          | 0/995 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/14 [00:00<?, ? examples/s]

Map:   0%|          | 0/28 [00:00<?, ? examples/s]

Map:   0%|          | 0/14 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/4163 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/1917 [00:00<?, ? examples/s]

### Fine-tuning

#### GPT-2

The first model we want to funetune is GPT2, which is an instance of a 
language model from hugging face. 

In [30]:
from LLMDialogue.models import GPT2

In [31]:
# Instantiate and load the model 
gpt2 = GPT2(seed=GLOBAL_SEED)
gpt2.load()
gpt2.to(GLOBAL_DEVICE)
print(gpt2)

Base model: GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50258, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D()
          (c_proj): Conv1D()
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D()
          (c_proj): Conv1D()
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50258, bias=False)
)
Base tokenizer: GPT2TokenizerFast(name_or_path='gpt2', vocab_size=50257, model_max_length=1024, is_f

In [32]:
# Now, finetune the model using the appropriate data module 
# Load the dm and prepare it.
dm = ICCHFLMDM(
    dset_variant="labels", 
    tokenizer=gpt2.tokenizer, 
    save_dir=OUTPUT_DIR
)


In [33]:
# Now, finetune the model. 
# gpt2.finetune(
#     dm=dm, 
#     save_dir=f"{OUTPUT_DIR}/models/gpt2",
#     num_train_epochs=1
# )

### TurnGPT

In [34]:
from LLMDialogue.models import TurnGPT 


In [35]:
turngpt = TurnGPT(seed=GLOBAL_SEED)
turngpt.load(model_head="LMHead")
turngpt.to(device=GLOBAL_DEVICE)
print(turngpt)

/opt/miniconda3/envs/gpt/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Additional special tokens map:
	bos_token: <|endoftext|>
	eos_token: <ts>
	unk_token: <|endoftext|>
	pad_token: <|endoftext|>
	additional_special_tokens: ['<SP1>', '<SP2>']

NOTE: This tokenizer assumes that there are 2 speakers
Initialized special wte for <ts> to average of ['!', '?', '.']
This will make the <ts> semantically closer to ['!', '?', '.']
Base model: TurnGPTLMHeadModel(
  (_transformer): GPT2LMHeadModel(
    (transformer): GPT2Model(
      (wte): Embedding(50260, 768)
      (wpe): Embedding(1024, 768)
      (drop): Dropout(p=0.1, inplace=False)
      (h): ModuleList(
        (0-11): 12 x GPT2Block(
          (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (attn): GPT2Attention(
            (c_attn): Conv1D()
            (c_proj): Conv1D()
            (attn_dropout): Dropout(p=0.1, inplace=False)
            (resid_dropout): Dropout(p=0.1, inplace=False)
          )
          (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
          (m

In [36]:
# Load the appropriate dm and finetune
dm = ICCSpokenLanguageModelingDM(
    dset_variant="labels", 
    tokenizer=tokenizer,
    save_dir=OUTPUT_DIR
)

Map:   0%|          | 0/21159 [00:00<?, ? examples/s]

Map:   0%|          | 0/743 [00:00<?, ? examples/s]

Map:   0%|          | 0/782 [00:00<?, ? examples/s]

Map:   0%|          | 0/802 [00:00<?, ? examples/s]

Map:   0%|          | 0/995 [00:00<?, ? examples/s]

...

Map:   0%|          | 0/591 [00:00<?, ? examples/s]

Map:   0%|          | 0/1066 [00:00<?, ? examples/s]

Map:   0%|          | 0/700 [00:00<?, ? examples/s]

Map:   0%|          | 0/859 [00:00<?, ? examples/s]

Map:   0%|          | 0/715 [00:00<?, ? examples/s]

In [37]:
# turngpt.finetune(
#     dm=dm,
#     save_dir=f"{OUTPUT_DIR}/models/turngpt",
#     max_epochs=1
# )

### Pipelines

In [38]:
from LLMDialogue.pipelines import CPPipelineConfig, CPPipeline

In [39]:
gpt2 = GPT2(seed=GLOBAL_SEED)
gpt2.load()

In [40]:
# Create the pipeline config 
config = CPPipelineConfig(
    model = gpt2, 
    utterances=['this is is a text'],
    N = -1,
    context_buffer_size=512,   
)

In [42]:
# Now, init the pipeline 
pipe = CPPipeline(config)
pipe.run()



Processing turns: 100%|██████████| 1/1 [00:07<00:00,  7.87s/it]


In [45]:
df = pd.DataFrame(pipe.info().data)
df

,turn_no,word_no,context,word,last_word_prob
0,0,1,this,is,7.019224e-14
1,0,2,this is,is,0.000000e+00
2,0,3,this is is,a,8.809402e-37
3,0,4,this is is a,text,0.000000e+00


In [46]:
from transformers import AutoModelForCausalLM, AutoTokenizer

In [52]:
model = AutoModelForCausalLM.from_pretrained("gpt2")
tokenizer = AutoTokenizer.from_pretrained("gpt2")
word = "this"

In [55]:
toks = tokenizer.encode(word,return_tensors="pt")
model(toks)

CausalLMOutputWithCrossAttentions(loss=None, logits=tensor([[[-31.8387, -30.9853, -34.4230,  ..., -39.7514, -38.6847, -32.3073]]],
       grad_fn=<UnsafeViewBackward0>), past_key_values=((tensor([[[[-1.1436e+00,  2.0865e+00,  1.0915e+00,  5.2886e-01,  6.7749e-01,
            2.1998e-01,  3.3289e-01,  5.0914e-01, -1.5674e+00,  7.3908e-01,
           -5.5447e-01, -4.3670e-01,  6.7484e-01,  7.6756e-02, -3.3582e-01,
           -1.0729e-01,  5.1017e-01, -8.2771e-02,  2.6644e+00, -8.4814e-03,
           -2.0417e+00,  9.1725e-02, -6.9554e-01, -1.3793e+00, -2.0295e-01,
           -8.7571e-01, -4.5975e-01, -2.4336e-01, -3.8781e-01, -7.2261e-01,
            1.1481e+00, -1.0246e+00, -1.0068e+00,  3.3189e-01, -2.0250e-01,
            2.5039e-01,  1.0650e+00,  8.2601e-01,  1.6686e-01,  4.5301e-01,
           -2.1656e-01,  3.3721e-04, -8.5817e-01,  2.5071e-02, -3.5411e-01,
           -2.8066e-01,  7.6596e-01,  2.4335e-01, -6.4883e-01,  2.7056e-01,
            7.4140e-01, -4.9807e-01,  9.9808e-01,  3